# 01 - Business Understanding

**Objective of this notebook:** define the target, the circumstances and what counts as a performance loss
*before* touching the data - as the course instruction demands: *"Start always with definition of targets and
circumstances (business understanding) before getting in touch with 'real' data or any analysis models."*

This notebook contains no data analysis on purpose. It produces the definitions, hypotheses and evidence rules
that notebooks 02-06 test. It combines Phase 1 of the team's first analysis (v1, section 3, original text) with
the exact definitions of the course slides and the quality requirements for the analysis.

| Notebook | CRISP-DM phase | Question |
|---|---|---|
| 01 | Business understanding | what is a performance loss, what must be shown? |
| 02 | Data understanding | what is in the recordings, what do the signals mean? (team quality checks) |
| 03 | Data preparation | one time base, stop windows, pallet cycles, activations (team pre-processing) |
| 04 | Modeling - Model 1 | engineering rule: which steps took clearly too long, and why? (team v1 analysis) |
| 05 | Modeling - Model 2 | does machine learning find losses the rule misses? (team feature table) |
| 06 | Evaluation + deployment | cycle-level effect, performance, evidence register, final result, reproducibility |

Run the notebooks in order 01-06; each one hands its results to the next through `data/processed/`. `python run_mp1.py`
optionally re-runs notebooks 02-06 without Jupyter (same figures and evidence tables).

## 1 The task

> "There are several performance loss reported (something between 1...10) in the production timeslot of the
> recorded data, try to find + allocate them to a physical equipment." (course slide, mini-project 1)

**Physical setup** (course slide): a small machine controlled by two PLCs - a Siemens S7 and a Beckhoff PLC, each
controlling about 50 % of the process: high-bay warehouse, conveyor, robot, drilling station, sorting station.

**What we must deliver for every loss:** when it happened, how long it lasted (or its production impact), which
physical equipment it belongs to, and the PLC evidence for all of this.

## 2 Definitions from the course material (OEE introduction)

OEE = Availability x Performance x Quality. The loss tree on the course slide assigns every loss to one factor:

| OEE factor | Loss group | Loss types on the slide |
|---|---|---|
| Availability | losses due to failures | disturbances; setting and adjustment |
| **Performance** | **losses in speed** | **idle time and short stops; reduced speed** |
| Quality | losses due to defects | bad parts; start-up losses |

Performance on the slide: '1/Theo. cycle time * Number of parts' over 'Up time'. Read literally this is not
dimensionless; we read it in the usual form **theoretical cycle time x number of parts / up time** (to be
confirmed - see open question A4).

**What follows for MP1**

1. A performance loss is **idle time / a short stop** or **reduced speed** inside production.
2. **Start-up losses belong to Quality**, not Performance -> the start-up of the recording is excluded.
3. The slide gives **no duration** that separates a short stop (performance) from a disturbance (availability).
   Whether the safety-curtain stops are performance or availability losses is therefore a definition question
   for the professor (A6); we treat them as short stops because they are short and ended by a simple reset.
4. The **theoretical cycle time is not given**. We must not invent it; OEE performance can only be bounded.
5. The course also frames the topic as *"detecting performance loss - short stops, cycle time variation"*: both
   individual short stops and the effect on the cycle time are in scope.

## 3 Phase 1 of the team's first analysis (v1, original text)

### Task (slide 10 of the lecture)

"There are several performance losses reported (between 1 and 10) in
the production timeslot of the recorded data. Find them and allocate
them to a physical equipment."

### What is a performance loss? (slide 4, OEE)

OEE = Availability x Performance x Quality. Performance losses are the
"losses in speed" during up time. They have two forms:

- **Idle time and short stops:** the machine waits, nothing is broken.
- **Reduced speed:** the machine works, but slower than normal.

### Working definition used in this notebook

The machine repeats the same pallet cycle. Each actuator output (motor,
valve, drill) is switched ON for a typical time in every cycle. A
performance loss is a step that takes **clearly longer than the same
step normally takes**. "Clearly longer" is defined statistically in
Phase 4.

### Constraints (what the data can and cannot answer)

- One recording of about 506 s, two PLCs (Siemens S7, Beckhoff).
- No theoretical cycle time is given, so an OEE performance value in %
  **cannot** be calculated. We measure lost seconds instead.
- No labels (nobody marked the losses), so supervised learning is not
  possible. The detection must be unsupervised.
- The Siemens sheet of the variable file describes every signal. The
  Beckhoff sheet gives **only the station**, no description.
- Both PLCs log irregularly (about every 76 ms and 93 ms) with gaps up
  to 1.84 s. Effects shorter than the largest gap cannot be told apart
  from a logging gap.

### Hypotheses, tests and decision rules (fixed before testing)

Data understanding

- **H1** The two PLC clocks agree, so events can be compared across
  PLCs. *Test:* edges of a signal documented as going from Siemens to
  Beckhoff. *Accept if* every edge difference is within one median
  logging step of each PLC.
- **H2** The machine repeats one cycle with the same steps. *Test:*
  steps per pallet cycle. *Accept if* every complete cycle contains
  the same steps.

Modelling

- **H3** Losses appear as single activations that are far longer than
  normal. *Test:* Tukey fence on the ON-duration of every output.
  *Accept if* 1 to 10 events are found and the count does not change
  between the fences k = 1.5 and k = 3.

Evaluation

- **H4** The detections are real, not logging gaps or longer travel.
  *Accept if* every detection stays too long after subtracting the
  timing uncertainty, and is longer than the longest normal activation
  by at least the detection limit.
- **H5** During the extra time the equipment stands still (short stop)
  rather than moves slowly. *Test:* encoder pulses, paired output
  signals, sensor arrival times. *Decided per event.*
- **H6** The stops have a cause that is visible in the data (safety
  light curtain or HMI stop). *Decided per event.*
- **H7** The events make the pallet cycle longer. *Test:* exact
  permutation test, cycles with versus without event. *Accept if*
  p < 0.05.
- **H8** Nothing is missed: every unusually long period without any
  switching lies inside an event or a normal movement.

## 4 Business objectives

| # | Objective | How we will know it is achieved |
|---|---|---|
| O1 | Find every performance loss in the production window | each event has start, end and evidence |
| O2 | Quantify each loss | seconds lost at the equipment, at cycle level, and line standstill |
| O3 | Allocate each loss to physical equipment | component + confidence + alternative explanations |
| O4 | Make sure the result is not an artefact | sensitivity to thresholds and to the time base; missed-event check |
| O5 | Test whether machine learning adds verifiable detections | Model 2 with its own business question |
| O6 | Reproducible, explainable delivery | notebooks 01-06 run in order, one evidence workbook |

The "1-10 losses" statement is a **plausibility check** only. A threshold is never tuned to hit that range.

## 5 Testable hypotheses

The course asks to *"formulate precise hypotheses that can be tested"*. H1-H8 come from the team's first
analysis (section 3, with their decision rules) and are kept; H9 and H10 were added.

| # | Hypothesis | Expected evidence if true | Notebook |
|---|---|---|---|
| H1 | The two PLC clocks agree within the logging resolution | a signal logged by both PLCs switches at the same time | 03 |
| H2 | The machine repeats one pallet cycle | the same sequence of warehouse, belt and robot steps in every cycle | 03 |
| H3 | A loss shows up as an actuator activation far longer than normal for the same output | a few runs far above the normal spread | 04 |
| H4 | These detections are real | not explained by a logging gap or by a longer normal move | 04 |
| H5 | The extra time is a standstill (short stop) | stop state with no actuator switching, motor OFF while requested, silent encoder | 04 |
| H6 | The cause is visible in the PLC data | a documented stop signal (safety curtain, HMI) | 04 |
| H7 | The events make the pallet cycle longer | event cycles longer than clean cycles at the same position | 06 |
| H8 | No stop is missed | every long period without any actuator activity is explained | 06 |
| H9 | Reduced speed exists | motion steps slower than their physical model, or drift | 04 |
| H10 | Machine learning finds losses the rule misses | ML candidates confirmed by independent PLC evidence | 05 |

## 6 What counts as evidence of a genuine loss

An unusual signal is **not** a performance loss. A candidate is accepted only if:

1. **Lost time:** a step took measurably longer than the normal duration of the same step, by more than logging
   inaccuracy can explain;
2. **Independent confirmation:** a signal that did *not* define the detection confirms it - e.g. a stop state
   during which no actuator switches, a light barrier reached late, a motor output OFF while its request is ON,
   a silent position encoder;
3. **In scope:** inside planned production (after start-up), not a normal mode difference (other rack position,
   other sorting lane), not commanded process dwell (oven time), not a station waiting for its next part.

Every candidate ends in one of these classes: *evidence-supported loss*, *uncertain*, *abnormal signal behaviour
but not a loss*, *anomaly but not a loss*, *explained*.

## 7 Quality requirements for this analysis

| Requirement | Where it is met |
|---|---|
| Never treat an ML parameter (e.g. contamination) as validation; sweep it | 05 |
| Check what a register does in normal operation before calling it a fault indicator | 02 |
| Distinguish "anomalous" from "performance loss" | 05, 06 |
| Compare ML with a simple rule-based baseline and test sensitivity | 04, 05, 06 |
| Keep timestamps and durations; do not delete repeated rows | 02, 03 |
| Translate PLC tags into named physical components | 02, 04 |
| Choose k (K-Means) by silhouette, not by habit | 05 |
| Quantify the uncertainty of any performance figure | 06 |
| Notebook and report numbers must come from one run | 06 (evidence workbook) |
| Concrete individual contributions per phase; AI-use disclosure | report: [TO BE COMPLETED] |

## 8 Scope and limitations known before the analysis

- One recording of about 8 minutes: few cycles, so every statistic is weak.
- No labels: nobody wrote down when the losses happened - validation has to use independent PLC signals.
- Beckhoff signals are documented by station only; meanings must be shown from behaviour.
- No theoretical cycle time; no quality data.

**Next:** notebook 02 looks at the data for the first time (the team's quality checks).